# Task 2: corruption classifier and hard-routed specialists (Kaggle run)

Same setup as the Task 1 notebook: GPU and Internet on, `WANDB_API_KEY` secret attached
(and `GITHUB_TOKEN` if the repository is private).

The classifier and the specialists are independent, so steps 4 and 5 can also be run
in two separate notebooks at the same time; step 6 needs both.

In [ ]:
# 1. Secrets -> environment variables
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "genai-a1"

REPO = "aleenababar04/genai-assignment-1"
try:
    token = secrets.get_secret("GITHUB_TOKEN")   # only needed for a private repo
    REPO_URL = f"https://{token}@github.com/{REPO}.git"
except Exception:
    REPO_URL = f"https://github.com/{REPO}.git"


In [ ]:
# 2. Get the code and install the few packages Kaggle does not ship
%cd /kaggle/working
!rm -rf genai-assignment-1
!git clone -q $REPO_URL genai-assignment-1
%cd genai-assignment-1
!git log --oneline -1
!pip install -q optuna wandb onnx onnxruntime pytorch-msssim python-dotenv

import torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Turn the GPU on in the notebook settings."


In [ ]:
# 3. Download Oxford-IIIT Pet and build the 128x128 cache (split checked against the repo)
!python -m src.data.prepare_pet
!python -m pytest -q


In [ ]:
# 4. Classifier: Optuna search, then full training -> checkpoints/task2_classifier.pt
!python -m src.training.train_classifier --mode optuna
!python -m src.training.train_classifier --mode final


In [ ]:
# 5. Specialists: one shared Optuna search, then each trained independently
#    -> checkpoints/task2_specialist_{salt_pepper,blur,occlusion}.pt
!python -m src.training.train_specialists --mode optuna
!python -m src.training.train_specialists --mode final


In [ ]:
# 6. Test evaluation (oracle and predicted routing) and ONNX export of all four models
!python -m src.evaluation.eval_task2
!python -m src.export.export_onnx --task task2


In [ ]:
# 7. Collect everything that must go back into the repository.
#    Download task2_outputs.zip from the notebook's Output panel.
!mkdir -p /kaggle/working/task2_outputs/configs
!cp configs/task2_*_best.yaml /kaggle/working/task2_outputs/configs/
!cp -r optuna_studies checkpoints models report/results report/figures /kaggle/working/task2_outputs/
%cd /kaggle/working
!zip -qr task2_outputs.zip task2_outputs
!ls -lh task2_outputs.zip
